In [1]:
from pathlib import Path
import random
import shutil
import hashlib
import csv
import pandas as pd

In [8]:
SEED = 42
TOTAL_CLASS = 200
TRAIN_PER_CLASS = 450
VAL_PER_CLASS = 100

assert (TRAIN_PER_CLASS + VAL_PER_CLASS == 550)

ROOT = "data/tiny_imagenet"
SOURCE_ROOT = Path(f"{ROOT}/tiny-imagenet-200")
DEST_ROOT = Path(f"{ROOT}/tiny-imagenet-200-{TRAIN_PER_CLASS}train-{VAL_PER_CLASS}val")


# Audit first if tain and test have unique images

In [3]:
def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


with open(SOURCE_ROOT / "wnids.txt", "r") as f:
    WNIDS = [line.strip() for line in f if line.strip()]

rows = []

for wnid in WNIDS:
    train_paths = sorted((SOURCE_ROOT / "train" / wnid / "images").glob("*.JPEG"))
    val_paths = sorted((SOURCE_ROOT / "val" / wnid / "images").glob("*.JPEG"))

    train_hashes = {sha256_file(path) for path in train_paths}
    val_hashes = {sha256_file(path) for path in val_paths}

    all_unique_hashes = train_hashes.union(val_hashes)
    cross_overlap = train_hashes.intersection(val_hashes)

    total_unique_images = len(all_unique_hashes)
    required_unique_images = TRAIN_PER_CLASS + VAL_PER_CLASS
    max_train_if_target_val = total_unique_images - VAL_PER_CLASS
    can_support_requested_split = total_unique_images >= required_unique_images

    rows.append({
        "wnid": wnid,
        "train_files": len(train_paths),
        "unique_train": len(train_hashes),
        "val_files": len(val_paths),
        "unique_val": len(val_hashes),
        "train_val_duplicate_hashes": len(cross_overlap),
        "total_unique_images": total_unique_images,
        "required_unique_images": required_unique_images,
        "max_train_if_val_target": max_train_if_target_val,
        "can_support_450_100": can_support_requested_split
    })

audit_df = pd.DataFrame(rows)

display(audit_df.sort_values("total_unique_images"))

minimum_unique = audit_df["total_unique_images"].min()
maximum_safe_train = audit_df["max_train_if_val_target"].min()
unsupported_classes = audit_df[~audit_df["can_support_450_100"]]

print("Requested train/class:", TRAIN_PER_CLASS)
print("Requested val/class:", VAL_PER_CLASS)
print("Required unique images/class:", TRAIN_PER_CLASS + VAL_PER_CLASS)
print("Minimum unique images in any class:", minimum_unique)
print("Maximum safe uniform train/class with val target:", maximum_safe_train)
print("Classes unable to support requested split:", len(unsupported_classes))

if len(unsupported_classes) == 0:
    print()
    print("All classes support the requested split.")
    print(f"Train: {TRAIN_PER_CLASS} images/class")
    print(f"Validation: {VAL_PER_CLASS} images/class")
    print(f"Total train images: {TRAIN_PER_CLASS * len(WNIDS)}")
    print(f"Total validation images: {VAL_PER_CLASS * len(WNIDS)}")
else:
    print()
    print("Some classes cannot support the requested split.")
    display(unsupported_classes)

,wnid,train_files,unique_train,val_files,unique_val,train_val_duplicate_hashes,total_unique_images,required_unique_images,max_train_if_val_target,can_support_450_100
0,n02124075,500,500,50,50,0,550,550,450,True
127,n03930313,500,500,50,50,0,550,550,450,True
128,n04118538,500,500,50,50,0,550,550,450,True
129,n04179913,500,500,50,50,0,550,550,450,True
130,n04311004,500,500,50,50,0,550,550,450,True
...,...,...,...,...,...,...,...,...,...,...
70,n02988304,500,500,50,50,0,550,550,450,True
71,n04398044,500,500,50,50,0,550,550,450,True
72,n02843684,500,500,50,50,0,550,550,450,True
62,n04456115,500,500,50,50,0,550,550,450,True


Requested train/class: 450
Requested val/class: 100
Required unique images/class: 550
Minimum unique images in any class: 550
Maximum safe uniform train/class with val target: 450
Classes unable to support requested split: 0

All classes support the requested split.
Train: 450 images/class
Validation: 100 images/class
Total train images: 90000
Total validation images: 20000


# Split Dataset to class 200 each to 450train 100val image

In [4]:
RECREATE_DEST = True


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


def read_box_map(path):
    box_map = {}

    if not path.exists():
        return box_map

    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line:
                continue

            filename = line.split("\t")[0]
            box_map[filename] = line

    return box_map


# ---------------------------------------------------------
# Safety
# ---------------------------------------------------------

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE_ROOT}")

if DEST_ROOT.exists():
    if RECREATE_DEST:
        print("Removing previous destination:", DEST_ROOT)
        shutil.rmtree(DEST_ROOT)
    else:
        raise FileExistsError(f"Destination already exists: {DEST_ROOT}")

# ---------------------------------------------------------
# Read classes
# ---------------------------------------------------------

with open(SOURCE_ROOT / "wnids.txt", "r") as f:
    WNIDS = [line.strip() for line in f if line.strip()]

if len(WNIDS) != TOTAL_CLASS:
    raise ValueError(f"Expected {TOTAL_CLASS} classes, found {len(WNIDS)}")

print("Classes:", len(WNIDS))

# ---------------------------------------------------------
# Create destination
# ---------------------------------------------------------

DEST_ROOT.mkdir(parents=True)
(DEST_ROOT / "train").mkdir()
(DEST_ROOT / "val").mkdir()

shutil.copy2(SOURCE_ROOT / "wnids.txt", DEST_ROOT / "wnids.txt")
shutil.copy2(SOURCE_ROOT / "words.txt", DEST_ROOT / "words.txt")

# ---------------------------------------------------------
# Manifest
# ---------------------------------------------------------

manifest_rows = []

# ---------------------------------------------------------
# Build each class independently
# ---------------------------------------------------------

for class_index, wnid in enumerate(WNIDS):
    source_train_class = SOURCE_ROOT / "train" / wnid
    source_val_class = SOURCE_ROOT / "val" / wnid

    source_train_images_dir = source_train_class / "images"
    source_val_images_dir = source_val_class / "images"

    source_train_boxes = source_train_class / f"{wnid}_boxes.txt"
    source_val_boxes = source_val_class / f"{wnid}_boxes.txt"

    dest_train_class = DEST_ROOT / "train" / wnid
    dest_val_class = DEST_ROOT / "val" / wnid

    dest_train_images_dir = dest_train_class / "images"
    dest_val_images_dir = dest_val_class / "images"

    dest_train_images_dir.mkdir(parents=True)
    dest_val_images_dir.mkdir(parents=True)

    dest_train_boxes = dest_train_class / f"{wnid}_boxes.txt"
    dest_val_boxes = dest_val_class / f"{wnid}_boxes.txt"

    train_paths = sorted(source_train_images_dir.glob("*.JPEG"))
    val_paths = sorted(source_val_images_dir.glob("*.JPEG"))

    train_box_map = read_box_map(source_train_boxes)
    val_box_map = read_box_map(source_val_boxes)

    # -----------------------------------------------------
    # Deduplicate original validation WITHIN THIS CLASS
    # -----------------------------------------------------

    unique_original_val = []
    class_val_hashes = set()

    for image_path in val_paths:
        image_hash = sha256_file(image_path)

        if image_hash in class_val_hashes:
            continue

        class_val_hashes.add(image_hash)
        unique_original_val.append((image_path, image_hash))

    # -----------------------------------------------------
    # Deduplicate original training WITHIN THIS CLASS
    # and remove any image already present in THIS CLASS val
    # -----------------------------------------------------

    unique_train_candidates = []
    class_train_hashes = set()

    for image_path in train_paths:
        image_hash = sha256_file(image_path)

        if image_hash in class_train_hashes:
            continue

        class_train_hashes.add(image_hash)

        if image_hash in class_val_hashes:
            continue

        unique_train_candidates.append((image_path, image_hash))

    # -----------------------------------------------------
    # Check enough unique images exist
    # -----------------------------------------------------

    additional_val_needed = VAL_PER_CLASS - len(unique_original_val)

    if additional_val_needed < 0:
        raise ValueError(
            f"{wnid}: original validation already has "
            f"{len(unique_original_val)} unique images, "
            f"which exceeds target {VAL_PER_CLASS}"
        )

    required_from_train = TRAIN_PER_CLASS + additional_val_needed

    if len(unique_train_candidates) < required_from_train:
        raise ValueError(
            f"\n{wnid}: NOT ENOUGH UNIQUE IMAGES\n"
            f"Unique original validation: {len(unique_original_val)}\n"
            f"Unique usable training: {len(unique_train_candidates)}\n"
            f"Need from train for validation: {additional_val_needed}\n"
            f"Need final training: {TRAIN_PER_CLASS}\n"
            f"Total needed from original training: {required_from_train}\n"
        )

    # -----------------------------------------------------
    # Deterministic split
    # -----------------------------------------------------

    rng = random.Random(SEED + class_index)
    rng.shuffle(unique_train_candidates)

    additional_val = unique_train_candidates[:additional_val_needed]
    selected_train = unique_train_candidates[additional_val_needed:additional_val_needed + TRAIN_PER_CLASS]

    selected_val = unique_original_val + additional_val

    # -----------------------------------------------------
    # Strict class-level uniqueness checks
    # -----------------------------------------------------

    selected_train_hashes = {image_hash for _, image_hash in selected_train}
    selected_val_hashes = {image_hash for _, image_hash in selected_val}

    if len(selected_train) != TRAIN_PER_CLASS:
        raise RuntimeError(f"{wnid}: train count is not {TRAIN_PER_CLASS}")

    if len(selected_val) != VAL_PER_CLASS:
        raise RuntimeError(f"{wnid}: val count is not {VAL_PER_CLASS}")

    if len(selected_train_hashes) != TRAIN_PER_CLASS:
        raise RuntimeError(f"{wnid}: duplicate image content inside TRAIN")

    if len(selected_val_hashes) != VAL_PER_CLASS:
        raise RuntimeError(f"{wnid}: duplicate image content inside VAL")

    overlap = selected_train_hashes.intersection(selected_val_hashes)

    if overlap:
        raise RuntimeError(
            f"{wnid}: identical image content exists in both TRAIN and VAL"
        )

    # -----------------------------------------------------
    # Copy TRAIN
    # -----------------------------------------------------

    train_box_lines = []

    for image_path, image_hash in selected_train:
        destination = dest_train_images_dir / image_path.name

        shutil.copy2(image_path, destination)

        if image_path.name not in train_box_map:
            raise ValueError(
                f"{wnid}: missing TRAIN bounding box for {image_path.name}"
            )

        train_box_lines.append(train_box_map[image_path.name])

        manifest_rows.append([
            wnid,
            image_path.name,
            "train",
            "original_train",
            image_hash
        ])

    # -----------------------------------------------------
    # Copy VALIDATION
    # -----------------------------------------------------

    val_box_lines = []

    for image_path, image_hash in selected_val:
        destination = dest_val_images_dir / image_path.name

        if destination.exists():
            raise FileExistsError(
                f"{wnid}: duplicate filename in destination validation: "
                f"{image_path.name}"
            )

        shutil.copy2(image_path, destination)

        if image_path.parent == source_val_images_dir:
            if image_path.name not in val_box_map:
                raise ValueError(
                    f"{wnid}: missing VAL bounding box for {image_path.name}"
                )

            val_box_lines.append(val_box_map[image_path.name])
            source_split = "original_val"

        else:
            if image_path.name not in train_box_map:
                raise ValueError(
                    f"{wnid}: missing TRAIN bounding box for {image_path.name}"
                )

            val_box_lines.append(train_box_map[image_path.name])
            source_split = "train_to_val"

        manifest_rows.append([
            wnid,
            image_path.name,
            "val",
            source_split,
            image_hash
        ])

    # -----------------------------------------------------
    # Write bounding-box files
    # -----------------------------------------------------

    with open(dest_train_boxes, "w") as f:
        for line in train_box_lines:
            f.write(line + "\n")

    with open(dest_val_boxes, "w") as f:
        for line in val_box_lines:
            f.write(line + "\n")

    print(
        f"{class_index + 1:03d}/{TOTAL_CLASS} | "
        f"{wnid} | "
        f"Train: {len(selected_train)} unique | "
        f"Val: {len(selected_val)} unique | "
        f"Overlap: {len(overlap)}"
    )

# ---------------------------------------------------------
# Copy TEST completely unchanged
# ---------------------------------------------------------

shutil.copytree(
    SOURCE_ROOT / "test",
    DEST_ROOT / "test"
)

# ---------------------------------------------------------
# Save manifest
# ---------------------------------------------------------

manifest_path = DEST_ROOT / "split_manifest.csv"

with open(manifest_path, "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "wnid",
        "filename",
        "new_split",
        "source_split",
        "sha256"
    ])

    writer.writerows(manifest_rows)

# ---------------------------------------------------------
# Save split information
# ---------------------------------------------------------

with open(DEST_ROOT / "split_info.txt", "w") as f:
    f.write(f"SPLIT_SEED={SEED}\n")
    f.write(f"TRAIN_PER_CLASS={TRAIN_PER_CLASS}\n")
    f.write(f"VAL_PER_CLASS={VAL_PER_CLASS}\n")
    f.write(f"NUM_CLASSES={len(WNIDS)}\n")
    f.write("UNIQUENESS=SHA256_PER_CLASS\n")
    f.write("TRAIN_VAL_OVERLAP_ALLOWED=FALSE\n")
    f.write("TEST=UNCHANGED_COPY\n")

print()
print("=" * 80)
print("DATASET CREATED")
print("=" * 80)
print("Destination:", DEST_ROOT)
print("Train/class:", TRAIN_PER_CLASS)
print("Validation/class:", VAL_PER_CLASS)
print("Total train:", TRAIN_PER_CLASS * len(WNIDS))
print("Total validation:", VAL_PER_CLASS * len(WNIDS))
print("Test copied unchanged")

Classes: 200
001/200 | n02124075 | Train: 450 unique | Val: 100 unique | Overlap: 0
002/200 | n04067472 | Train: 450 unique | Val: 100 unique | Overlap: 0
003/200 | n04540053 | Train: 450 unique | Val: 100 unique | Overlap: 0
004/200 | n04099969 | Train: 450 unique | Val: 100 unique | Overlap: 0
005/200 | n07749582 | Train: 450 unique | Val: 100 unique | Overlap: 0
006/200 | n01641577 | Train: 450 unique | Val: 100 unique | Overlap: 0
007/200 | n02802426 | Train: 450 unique | Val: 100 unique | Overlap: 0
008/200 | n09246464 | Train: 450 unique | Val: 100 unique | Overlap: 0
009/200 | n07920052 | Train: 450 unique | Val: 100 unique | Overlap: 0
010/200 | n03970156 | Train: 450 unique | Val: 100 unique | Overlap: 0
011/200 | n03891332 | Train: 450 unique | Val: 100 unique | Overlap: 0
012/200 | n02106662 | Train: 450 unique | Val: 100 unique | Overlap: 0
013/200 | n03201208 | Train: 450 unique | Val: 100 unique | Overlap: 0
014/200 | n02279972 | Train: 450 unique | Val: 100 unique | Over

# Test DataLoader

In [5]:
print("=" * 80)
print("FINAL DATASET VERIFICATION")
print("=" * 80)


total_train = 0
total_val = 0


for class_index, wnid in enumerate(WNIDS):
    train_paths = sorted(
        (DEST_ROOT / "train" / wnid / "images").glob("*.JPEG")
    )

    val_paths = sorted(
        (DEST_ROOT / "val" / wnid / "images").glob("*.JPEG")
    )

    train_hashes = [sha256_file(path) for path in train_paths]
    val_hashes = [sha256_file(path) for path in val_paths]

    unique_train_hashes = set(train_hashes)
    unique_val_hashes = set(val_hashes)

    overlap = unique_train_hashes.intersection(unique_val_hashes)

    assert len(train_paths) == TRAIN_PER_CLASS, (
        f"{wnid}: train count = {len(train_paths)}"
    )

    assert len(val_paths) == VAL_PER_CLASS, (
        f"{wnid}: val count = {len(val_paths)}"
    )

    assert len(unique_train_hashes) == TRAIN_PER_CLASS, (
        f"{wnid}: duplicate image content inside TRAIN"
    )

    assert len(unique_val_hashes) == VAL_PER_CLASS, (
        f"{wnid}: duplicate image content inside VAL"
    )

    assert len(overlap) == 0, (
        f"{wnid}: TRAIN/VAL image-content overlap found"
    )

    total_train += len(train_paths)
    total_val += len(val_paths)

    print(
        f"{class_index + 1:03d}/{TOTAL_CLASS} | "
        f"{wnid} | "
        f"Train={len(train_paths)} | "
        f"Val={len(val_paths)} | "
        f"Overlap={len(overlap)}"
    )


source_test_paths = sorted(
    (SOURCE_ROOT / "test" / "images").glob("*.JPEG")
)

dest_test_paths = sorted(
    (DEST_ROOT / "test" / "images").glob("*.JPEG")
)


assert total_train == TOTAL_CLASS * TRAIN_PER_CLASS
assert total_val == TOTAL_CLASS * VAL_PER_CLASS

assert len(source_test_paths) == len(dest_test_paths)


print()
print("=" * 80)
print("VERIFICATION PASSED")
print("=" * 80)

print(f"{TOTAL_CLASS} classes")
print(f"{TRAIN_PER_CLASS} unique training images per class")
print(f"{VAL_PER_CLASS} unique validation images per class")
print("0 duplicate image content inside each TRAIN class")
print("0 duplicate image content inside each VAL class")
print("0 TRAIN ↔ VAL overlap within every class")
print(f"Total TRAIN = {total_train}")
print(f"Total VAL = {total_val}")
print("TEST copied unchanged")

FINAL DATASET VERIFICATION
001/200 | n02124075 | Train=450 | Val=100 | Overlap=0
002/200 | n04067472 | Train=450 | Val=100 | Overlap=0
003/200 | n04540053 | Train=450 | Val=100 | Overlap=0
004/200 | n04099969 | Train=450 | Val=100 | Overlap=0
005/200 | n07749582 | Train=450 | Val=100 | Overlap=0
006/200 | n01641577 | Train=450 | Val=100 | Overlap=0
007/200 | n02802426 | Train=450 | Val=100 | Overlap=0
008/200 | n09246464 | Train=450 | Val=100 | Overlap=0
009/200 | n07920052 | Train=450 | Val=100 | Overlap=0
010/200 | n03970156 | Train=450 | Val=100 | Overlap=0
011/200 | n03891332 | Train=450 | Val=100 | Overlap=0
012/200 | n02106662 | Train=450 | Val=100 | Overlap=0
013/200 | n03201208 | Train=450 | Val=100 | Overlap=0
014/200 | n02279972 | Train=450 | Val=100 | Overlap=0
015/200 | n02132136 | Train=450 | Val=100 | Overlap=0
016/200 | n04146614 | Train=450 | Val=100 | Overlap=0
017/200 | n07873807 | Train=450 | Val=100 | Overlap=0
018/200 | n02364673 | Train=450 | Val=100 | Overlap=0
0

In [10]:
from src.dataset.tiny_imagenet_dataset_2 import TinyImageNetLoader, get_tiny_imagenet_class_names, TINY_ROOT

train_dataset, val_dataset, test_dataset, dataset_name = TinyImageNetLoader(seed=SEED).load()
CLASS_NAMES = get_tiny_imagenet_class_names(train_dataset, tiny_root=TINY_ROOT)

print()
print("Dataset:", dataset_name)
print("Number of classes:", len(CLASS_NAMES))
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print()
print("Expected train:", TOTAL_CLASS * TRAIN_PER_CLASS)
print("Expected validation:", TOTAL_CLASS * VAL_PER_CLASS)

assert len(train_dataset) == TOTAL_CLASS * TRAIN_PER_CLASS
assert len(val_dataset) == TOTAL_CLASS * VAL_PER_CLASS
assert len(train_dataset.classes) == TOTAL_CLASS
assert train_dataset.class_to_idx == val_dataset.class_to_idx

print()
print("Loader is using the new 450/100 dataset correctly.")

Tiny ImageNet loaded
Root: data/tiny_imagenet/tiny-imagenet-200-450train-100val
Classes: 200
Train images: 90000
Validation images: 20000
Test images: 10000

Dataset: tiny_imagenet
Number of classes: 200
Train: 90000
Validation: 20000
Test: 10000

Expected train: 90000
Expected validation: 20000

Loader is using the new 450/100 dataset correctly.


# Get Mean and Std

In [2]:
from src.dataset.tiny_imagenet_dataset_2 import TinyImageNetDataset, convert_to_rgb, TINY_ROOT
import torch

from torch.utils.data import DataLoader
from torchvision import transforms


stats_transform = transforms.Compose([
    transforms.Lambda(convert_to_rgb),
    transforms.ToTensor()
])


stats_dataset = TinyImageNetDataset(
    root=TINY_ROOT,
    split="train",
    transform=stats_transform
)


stats_loader = DataLoader(
    stats_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=4
)


channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
num_pixels = 0


for images, _ in stats_loader:
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_squared_sum += (images ** 2).sum(dim=[0, 2, 3])
    num_pixels += images.size(0) * images.size(2) * images.size(3)


means = channel_sum / num_pixels
stds = torch.sqrt(channel_squared_sum / num_pixels - means ** 2)


print("Mean:", means.tolist())
print("Std:", stds.tolist())

Mean: [0.4803270399570465, 0.44804883003234863, 0.3975021243095398]
Std: [0.27645865082740784, 0.26883983612060547, 0.28148162364959717]
